In [1]:
import torch
from transformers import MusicgenMelodyForConditionalGeneration

model_name="facebook/musicgen-melody"
model = MusicgenMelodyForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
        ).to('cuda')
audio_encoder = model.audio_encoder

C:\Users\User\PycharmProjects\MTS\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\PycharmProjects\MTS\venv\lib\site-packages\torch\nn\utils\weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
C:\Users\User\PycharmProjects\MTS\venv\lib\site-packages\transformers\models\encodec\modeling_encodec.py:120: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("padding_total", torch.tensor(kernel_size - stride, dtype=torch.int64), persistent=False)


In [3]:
from datasets import load_dataset, Features, Value, Sequence

dataset_path = "../output/Lora/dataset"
features = Features({
            'text': Value('string'),
            'input_ids': Sequence(Value('int32')),
            'attention_mask': Sequence(Value('int8')),
            'input_audio_values': Sequence(Value('float32')),
            'target_audio_values': Sequence(Value('float32'))
        })

dataset = load_dataset(path=dataset_path, features=features, split='train', streaming=True)
data = dataset.filter(lambda example, idx: len(example['input_audio_values']) > 0, with_indices=True)

In [3]:
from transformers import AutoProcessor
from training.dataset import create_musicgen_dataset

test_dataset = create_musicgen_dataset(
        data,
        processor=AutoProcessor.from_pretrained('facebook/musicgen-melody'),
    )

NameError: name 'data' is not defined

In [5]:
i = None
for example in test_dataset:
    i = example
    break

In [14]:
encoded1 = audio_encoder.encode(
                i['labels'].to(torch.float16).unsqueeze(0).to('cuda')
            )
encoded2 = audio_encoder.encode(
                i['labels'][:32000*10].to(torch.float16).unsqueeze(0).to('cuda')
            )

In [19]:
encoded1.audio_codes.shape

torch.Size([1, 1, 4, 1500])

In [20]:
encoded2.audio_codes.shape

torch.Size([1, 1, 4, 1500])

In [1]:
import torch
from model.mts_generate import MTSGen, MTSGenConfig

config = MTSGenConfig(
    hidden_size=512,
    num_hidden_layers=4,
    num_attention_heads=8,
    num_durations=13,
    num_techniques=14,
    context_bars=4,
    predict_bars=1,
    max_fret=24,
    freeze_encoder=True
)

# 创建模型
model = MTSGen(config)
print(f"模型参数总数: {sum(p.numel() for p in model.parameters()):,}")
print(f"可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 测试数据
batch_size = 2
audio_length = 16000 * 5
context_length = 64

# 模拟音频输入
dummy_audio = torch.randn(batch_size, audio_length)

# 模拟上下文音符
dummy_context = {
    'duration': torch.randint(0, 13, (batch_size, context_length)),
    'fret': torch.randint(0, 26, (batch_size, context_length, 6)),
    'technique': torch.randint(0, 14, (batch_size, context_length, 6))
}

# 模拟目标音符
target_length = 16
dummy_target = {
    'duration': torch.randint(0, 13, (batch_size, target_length)),
    'fret': torch.randint(0, 26, (batch_size, target_length, 6)),
    'technique': torch.randint(0, 14, (batch_size, target_length, 6))
}

C:\Users\User\PycharmProjects\MTS\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


加载音频编码器: facebook/wav2vec2-base-960h


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\User\PycharmProjects\MTS\venv\lib\site-packages\torch\nn\modules\transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


NoteEmbeddingSimple维度: total=512, hidden_size=512
模型初始化完成，每根弦独立输出品位和技巧
模型参数总数: 109,875,197
可训练参数: 15,503,485


In [2]:
print("\n测试生成模式:")
with torch.no_grad():
    generate_outputs = model(
        audio_input=dummy_audio,
        context_notes=dummy_context,
        teacher_forcing=False,
        generate_length=64
    )

    for key, value in generate_outputs.items():
        if value is not None:
            print(f"  {key}: {value.shape}")


测试生成模式:
  duration: torch.Size([2, 64])
  fret: torch.Size([2, 64, 6])
  technique: torch.Size([2, 64, 6])


In [3]:
sample = {}
for key, value in generate_outputs.items():
        if value is not None:
            sample[key] = value[0]

In [5]:
from model.dataset import decode

song = decode(sample)

TypeError: both arguments should be Rational instances

In [14]:
from torch import Tensor

len(sample['duration'])
Tensor(1)[0].tolist()

1.401298464324817e-45